# Adam Optimizer — Section 9

**Adam** (Adaptive Moment Estimation) combines momentum and RMSProp with bias correction:

$$m_t = \beta_1 m_{t-1} + (1-\beta_1)\, g_t \quad \text{(1st moment — momentum)}$$
$$v_t = \beta_2 v_{t-1} + (1-\beta_2)\, g_t^2 \quad \text{(2nd moment — variance)}$$
$$\hat{m}_t = \frac{m_t}{1-\beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1-\beta_2^t} \quad \text{(bias correction)}$$
$$\theta \leftarrow \theta - \frac{\alpha}{\sqrt{\hat{v}_t} + \varepsilon}\, \hat{m}_t$$

Default values: $\beta_1 = 0.9$, $\beta_2 = 0.999$, $\varepsilon = 10^{-8}$, $\alpha = 0.001$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets


def get_loss(surface, kappa):
    if surface == 'bowl':
        b = 1 / np.sqrt(kappa)
        return (lambda x, y: 0.5*x**2 + 0.5*(y/b)**2,
                lambda x, y: x, lambda x, y: y/b**2,
                -1.8, -1.8, (-2.2, 2.2), (-2.2, 2.2))
    elif surface == 'banana':
        return (lambda x, y: (1-x)**2 + 100*(y-x**2)**2,
                lambda x, y: -2*(1-x)-400*x*(y-x**2), lambda x, y: 200*(y-x**2),
                -1.2, 1.0, (-2.0, 2.0), (-1.0, 3.0))
    else:
        return (lambda x, y: x**2/kappa - y**2 + 0.1*y**4,
                lambda x, y: 2*x/kappa, lambda x, y: -2*y+0.4*y**3,
                0.5, 0.2, (-3.0, 3.0), (-2.5, 2.5))


def run_adam(f, gx_fn, gy_fn, x0, y0, steps, lr=0.01, b1=0.9, b2=0.999, eps=1e-8):
    x, y = float(x0), float(y0)
    m1x, m1y = 0.0, 0.0
    m2x, m2y = 1e-8, 1e-8
    path, losses, m1_hist, m2_hist, eff_hist = [(x, y)], [f(x, y)], [], [], []

    for t in range(1, steps + 1):
        gx, gy = gx_fn(x, y), gy_fn(x, y)
        m1x = b1*m1x + (1-b1)*gx; m1y = b1*m1y + (1-b1)*gy
        m2x = b2*m2x + (1-b2)*gx**2; m2y = b2*m2y + (1-b2)*gy**2
        m1xh = m1x / (1 - b1**t); m1yh = m1y / (1 - b1**t)
        m2xh = m2x / (1 - b2**t); m2yh = m2y / (1 - b2**t)
        x = np.clip(x - lr * m1xh / (np.sqrt(m2xh) + eps), -5, 5)
        y = np.clip(y - lr * m1yh / (np.sqrt(m2yh) + eps), -5, 5)
        path.append((x, y)); losses.append(max(1e-10, f(x, y)))
        m1_hist.append((m1xh, m1yh))
        m2_hist.append((m2xh, m2yh))
        eff_hist.append((lr / (np.sqrt(m2xh) + eps), lr / (np.sqrt(m2yh) + eps)))
    return np.array(path), np.array(losses), np.array(m1_hist), np.array(m2_hist), np.array(eff_hist)


def run_simple(name, f, gx_fn, gy_fn, x0, y0, steps, lr, beta=0.9, rho=0.9):
    x, y = float(x0), float(y0)
    vx, vy, Ex, Ey = 0.0, 0.0, 1e-8, 1e-8
    path, losses = [(x, y)], [f(x, y)]
    for _ in range(steps):
        gx, gy = gx_fn(x, y), gy_fn(x, y)
        if name == 'GD': dx, dy = -lr*gx, -lr*gy
        elif name == 'Mom': vx=beta*vx-lr*gx; vy=beta*vy-lr*gy; dx, dy = vx, vy
        else: Ex=rho*Ex+(1-rho)*gx**2; Ey=rho*Ey+(1-rho)*gy**2; dx=-lr/np.sqrt(Ex+1e-8)*gx; dy=-lr/np.sqrt(Ey+1e-8)*gy
        x, y = np.clip(x+dx,-5,5), np.clip(y+dy,-5,5)
        path.append((x, y)); losses.append(max(1e-10, f(x, y)))
    return np.array(path), np.array(losses)


def draw_adam(surface='banana', kappa=10, steps=300,
              lr_adam=0.01, b1=0.9, b2=0.999,
              lr_gd=0.03, lr_mom=0.03, lr_rms=0.03,
              show_gd=True, show_mom=True, show_rms=True):

    f, gx_fn, gy_fn, x0, y0, xlim, ylim = get_loss(surface, kappa)

    adam_path, adam_loss, m1h, m2h, effh = run_adam(f, gx_fn, gy_fn, x0, y0, steps, lr_adam, b1, b2)
    results = {'Adam': (adam_path, adam_loss)}
    colors = {'Adam':'#7c3aed','GD':'#2563eb','Momentum':'#dc2626','RMSProp':'#d97706'}

    if show_gd:  results['GD']       = run_simple('GD',  f, gx_fn, gy_fn, x0, y0, steps, lr_gd)
    if show_mom: results['Momentum'] = run_simple('Mom', f, gx_fn, gy_fn, x0, y0, steps, lr_mom)
    if show_rms: results['RMSProp']  = run_simple('RMS', f, gx_fn, gy_fn, x0, y0, steps, lr_rms)

    xs = np.linspace(*xlim, 200); ys = np.linspace(*ylim, 200)
    XX, YY = np.meshgrid(xs, ys)
    ZZ = np.clip(f(XX, YY), None, 50)

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))

    # Contour
    axes[0, 0].contourf(XX, YY, ZZ, levels=20, cmap='RdBu_r', alpha=0.5)
    for name, (path, _) in results.items():
        axes[0, 0].plot(path[:, 0], path[:, 1], '-', color=colors[name], lw=2, label=name)
        axes[0, 0].plot(path[-1, 0], path[-1, 1], 'o', color=colors[name], ms=6)
    axes[0, 0].scatter([x0], [y0], c='k', s=80, marker='*', zorder=5)
    axes[0, 0].set_xlim(xlim); axes[0, 0].set_ylim(ylim)
    axes[0, 0].set_title('Trajectories', fontsize=10); axes[0, 0].legend(fontsize=8)

    # Loss curves
    for name, (_, losses) in results.items():
        axes[0, 1].semilogy(losses, color=colors[name], lw=2, label=f'{name}')
    axes[0, 1].set_title('Convergence (log loss)', fontsize=10)
    axes[0, 1].grid(True, which='both', alpha=0.3); axes[0, 1].legend(fontsize=8)

    # Adam internals
    t_ax = np.arange(1, steps + 1)
    axes[0, 2].plot(t_ax, m1h[:, 0], color='#7c3aed', lw=2, label='dim 1')
    axes[0, 2].plot(t_ax, m1h[:, 1], color='#be185d', lw=2, ls='--', label='dim 2')
    axes[0, 2].set_title('1st moment m̂ₜ (bias-corrected)', fontsize=10)
    axes[0, 2].legend(fontsize=8); axes[0, 2].grid(True, alpha=0.3)

    axes[1, 0].semilogy(t_ax, m2h[:, 0], color='#7c3aed', lw=2, label='dim 1')
    axes[1, 0].semilogy(t_ax, m2h[:, 1], color='#be185d', lw=2, ls='--', label='dim 2')
    axes[1, 0].set_title('2nd moment v̂ₜ (variance estimate)', fontsize=10)
    axes[1, 0].legend(fontsize=8); axes[1, 0].grid(True, which='both', alpha=0.3)

    axes[1, 1].plot(t_ax, effh[:, 0], color='#7c3aed', lw=2, label='dim 1')
    axes[1, 1].plot(t_ax, effh[:, 1], color='#be185d', lw=2, ls='--', label='dim 2')
    axes[1, 1].set_title('Effective step α/√v̂ₜ per dimension', fontsize=10)
    axes[1, 1].legend(fontsize=8); axes[1, 1].grid(True, alpha=0.3)

    axes[1, 2].axis('off')
    summary = '\n'.join([f'{n}: {l[-1]:.2e}' for n, (_, l) in results.items()])
    axes[1, 2].text(0.05, 0.9, f'Final losses:\n{summary}\n\nAdam = Momentum + RMSProp\n+ bias correction',
                    transform=axes[1, 2].transAxes, fontsize=11, va='top', family='monospace')

    plt.suptitle(f'Adam (α={lr_adam}, β₁={b1}, β₂={b2})  surface={surface}  κ={kappa}', fontsize=10)
    plt.tight_layout()
    plt.show()


widgets.interact(
    draw_adam,
    surface=widgets.Dropdown(options=['bowl', 'banana', 'saddle_valley'], value='banana', description='Surface'),
    kappa=widgets.IntSlider(min=1, max=50, step=1, value=10, description='κ', continuous_update=False),
    steps=widgets.IntSlider(min=50, max=1000, step=50, value=300, description='Steps', continuous_update=False),
    lr_adam=widgets.FloatLogSlider(min=-4, max=-1, step=0.25, value=0.01, description='α (Adam)', continuous_update=False),
    b1=widgets.FloatSlider(min=0.5, max=0.999, step=0.01, value=0.9, description='β₁', continuous_update=False),
    b2=widgets.FloatSlider(min=0.5, max=0.9999, step=0.001, value=0.999, description='β₂', continuous_update=False),
    lr_gd=widgets.FloatLogSlider(min=-3, max=-0.5, step=0.25, value=0.03, description='α (GD)', continuous_update=False),
    lr_mom=widgets.FloatLogSlider(min=-3, max=-0.5, step=0.25, value=0.03, description='α (Mom)', continuous_update=False),
    lr_rms=widgets.FloatLogSlider(min=-3, max=-0.5, step=0.25, value=0.03, description='α (RMS)', continuous_update=False),
    show_gd=widgets.Checkbox(value=True, description='Show GD'),
    show_mom=widgets.Checkbox(value=True, description='Show Mom'),
    show_rms=widgets.Checkbox(value=True, description='Show RMS'),
)